In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score


raw_df = pd.read_csv('f1_lap_model_data.csv')
track_df = pd.read_csv('track_char.csv')
track_df['EventName'] = track_df['EventName'].str.strip()
track_df['TrackDirection'] = track_df['TrackDirection'].str.strip().str.capitalize()

raw_df = raw_df.merge(track_df, on='EventName', how='left')

C:\Users\bhavi\AppData\Local\Temp\ipykernel_12220\864078984.py:6: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv('f1_lap_model_data.csv')


In [2]:
position_df = raw_df.sort_values(['EventName', 'Driver', 'LapNumber']).copy()

position_df['PrevPosition'] = position_df.groupby(['EventName', 'Driver'])['Position'].shift(1)
position_df['PrevLapWasPit'] = position_df.groupby(['EventName', 'Driver'])['is_pit_lap'].shift(1)

# Rolling recent pace (last 3 laps) per driver — this is the pace-trend signal that was missing
position_df['RecentPace'] = position_df.groupby(['EventName', 'Driver'])['LapTime_Seconds'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)

position_df['Overtook'] = (
    (position_df['Position'] < position_df['PrevPosition']) & position_df['PrevPosition'].notna()
).astype(int)

In [3]:
lap_pit_status = position_df.groupby(['EventName', 'LapNumber', 'Position'])['is_pit_lap'].first().reset_index()
position_df = position_df.merge(
    lap_pit_status.rename(columns={'Position': 'PrevPosition', 'is_pit_lap': 'CarAheadPitted'}),
    on=['EventName', 'LapNumber', 'PrevPosition'],
    how='left'
)

position_df['GenuineOvertake'] = (
    (position_df['Overtook'] == 1) &
    (position_df['is_pit_lap'] == False) &
    (position_df['PrevLapWasPit'].fillna(False) == False) &
    (position_df['CarAheadPitted'].fillna(False) == False)
)

print("Raw position gains:", position_df['Overtook'].sum())
print("Genuine overtakes after filtering:", position_df['GenuineOvertake'].sum())

Raw position gains: 25802
Genuine overtakes after filtering: 25802


C:\Users\bhavi\AppData\Local\Temp\ipykernel_12220\247194094.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (position_df['PrevLapWasPit'].fillna(False) == False) &
C:\Users\bhavi\AppData\Local\Temp\ipykernel_12220\247194094.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (position_df['CarAheadPitted'].fillna(False) == False)


In [4]:
car_ahead_stats = position_df[['EventName', 'LapNumber', 'Position', 'TyreLife', 'Compound', 'RecentPace']].rename(
    columns={
        'Position': 'PrevPosition',
        'TyreLife': 'CarAhead_TyreLife',
        'Compound': 'CarAhead_Compound',
        'RecentPace': 'CarAhead_RecentPace'
    }
)

position_df = position_df.merge(
    car_ahead_stats,
    on=['EventName', 'LapNumber', 'PrevPosition'],
    how='left'
)

# Relative (differential) features — the actual fix
position_df['TyreLife_Delta'] = position_df['TyreLife'] - position_df['CarAhead_TyreLife']       # negative = fresher tires than car ahead
position_df['Pace_Delta'] = position_df['RecentPace'] - position_df['CarAhead_RecentPace']        # negative = faster than car ahead
position_df['SameCompound'] = (position_df['Compound'] == position_df['CarAhead_Compound']).astype(int)


In [5]:
sample = position_df[(position_df['EventName'] == 'Bahrain Grand Prix')].dropna(
    subset=['TyreLife_Delta', 'Pace_Delta']
)
print(sample[['Driver', 'LapNumber', 'Position', 'PrevPosition', 
              'TyreLife_Delta', 'Pace_Delta', 'GenuineOvertake']].head(20))



      Driver  LapNumber  Position  PrevPosition  TyreLife_Delta  Pace_Delta  \
29337    ALB        2.0      14.0          14.0             0.0    0.000000   
29339    ALB        2.0      11.0          14.0             0.0   -3.404000   
29341    ALB        3.0      13.0          11.0             0.0    2.643333   
29342    ALB        3.0      13.0          11.0             0.0   -0.413333   
29343    ALB        3.0      13.0          11.0            -3.0    2.895000   
29344    ALB        3.0      11.0          13.0             0.0   -2.643333   
29345    ALB        3.0      11.0          13.0             0.0    0.521333   
29346    ALB        3.0      11.0          13.0             0.0    0.195333   
29347    ALB        3.0      12.0          11.0             0.0   -0.438000   
29348    ALB        3.0      12.0          11.0             0.0   -3.494667   
29349    ALB        3.0      12.0          11.0            -3.0   -0.186333   
29350    ALB        4.0      13.0          12.0     

In [6]:
model_df = raw_df.copy()
model_df['is_clean_lap'] = (model_df['is_green_flag'] & 
                              ~model_df['is_pit_lap'] & 
                              (model_df['IsAccurate'] == True))
model_df = model_df[model_df['is_clean_lap'] == True].copy()
model_df = model_df.dropna(subset=['LapTime_Seconds', 'TyreLife', 'TrackTemp', 'AirTemp', 'Position'])

model_df = model_df.merge(
    position_df[['EventName', 'Driver', 'LapNumber', 'GenuineOvertake', 
                 'TyreLife_Delta', 'Pace_Delta', 'SameCompound', 'GapToCarAhead_Secs']],
    on=['EventName', 'Driver', 'LapNumber'],
    how='left',
    suffixes=('', '_dup')
)
model_df['GenuineOvertake'] = model_df['GenuineOvertake'].fillna(False).astype(int)

print(model_df['GenuineOvertake'].value_counts())

GenuineOvertake
0    230059
1    163908
Name: count, dtype: int64


In [7]:
features = ['GapToCarAhead_Secs', 'TyreLife_Delta', 'Pace_Delta', 'SameCompound',
            'TrackTemp', 'AirTemp', 'CircuitLength_km', 'NumCorners', 
            'NumDRSZones', 'AvgSpeed_kmh', 'DownforceLevel']

X = model_df[features].copy()
y = model_df['GenuineOvertake']

X = pd.get_dummies(X, columns=['DownforceLevel'])
X = X.dropna()
y = y.loc[X.index]

print("Rows after dropna:", len(X))

Rows after dropna: 323455


In [8]:
sorted_events = model_df[['RoundNumber', 'EventName']].drop_duplicates().sort_values('RoundNumber')
event_order = sorted_events['EventName'].tolist()
n_test_races = 5
train_events = event_order[:-n_test_races]
test_events = event_order[-n_test_races:]

train_mask = model_df.loc[X.index, 'EventName'].isin(train_events)
test_mask = model_df.loc[X.index, 'EventName'].isin(test_events)

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [9]:
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, preds))
print("ROC-AUC:", roc_auc_score(y_test, probs))

              precision    recall  f1-score   support

           0       0.73      0.79      0.76     22532
           1       0.69      0.62      0.65     16856

    accuracy                           0.72     39388
   macro avg       0.71      0.70      0.71     39388
weighted avg       0.71      0.72      0.71     39388

ROC-AUC: 0.8109599343434442


In [10]:
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(15))
overtake_rows = model_df[model_df['GenuineOvertake'] == 1].sample(10, random_state=1)
print(overtake_rows[['Driver', 'LapNumber', 'Pace_Delta', 'TyreLife_Delta', 'GapToCarAhead_Secs']])

Pace_Delta               0.299852
GapToCarAhead_Secs       0.242693
TyreLife_Delta           0.194687
TrackTemp                0.115766
AirTemp                  0.089030
SameCompound             0.015104
CircuitLength_km         0.011307
AvgSpeed_kmh             0.010850
NumCorners               0.008703
NumDRSZones              0.005231
DownforceLevel_high      0.002539
DownforceLevel_medium    0.002287
DownforceLevel_low       0.001952
dtype: float64
       Driver  LapNumber  Pace_Delta  TyreLife_Delta  GapToCarAhead_Secs
21871     STR        6.0    1.383667             0.0               2.233
329254    ALO       63.0    0.813000             0.0              14.932
107465    STR       17.0    0.562333             3.0               3.122
220558    LEC       42.0    0.094667             4.0                 NaN
218871    OCO       38.0    0.507000           -15.0               0.297
61433     LEC        6.0   -0.546667             0.0               0.768
169137    ALB       28.0    0.55

In [11]:
import joblib
joblib.dump(model, 'over_take.joblib')
joblib.dump(X_train.columns, 'overtake_columns.joblib')

['overtake_columns.joblib']